# Trading App V2

This notebook is the live-trading shell for `optimal_trader` going forward.

- `quant-warehouse` owns historical refresh, point-in-time data reads, feature engineering, target engineering, and ThetaData option data.
- `quant-orchestrator` owns feature-family classifier training, option-ranker training, backtests, and strategy artifacts.
- `optimal_trader` owns live broker reconciliation, order planning, Streamlit leaderboard generation, and guarded order submission.

Default behavior builds plans and artifacts only. Orders are submitted exclusively from the generated Streamlit frontend after clicking the Submit Orders button.

In [ ]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
import json
import os
import sys

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 260)
load_dotenv(Path.cwd().resolve() / ".env", override=False)

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "app").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def prefer_local_repo(package_name: str, repo_dir: Path) -> None:
    repo_dir = repo_dir.resolve()
    if not repo_dir.exists():
        return
    repo_text = str(repo_dir)
    sys.path[:] = [
        entry
        for entry in sys.path
        if str(Path(entry or ".").expanduser().resolve()) != repo_text
    ]
    sys.path.insert(0, repo_text)
    loaded = sys.modules.get(package_name)
    loaded_file = Path(str(getattr(loaded, "__file__", ""))).resolve() if loaded is not None else None
    if loaded_file is not None and repo_dir not in loaded_file.parents:
        for module_name in [name for name in sys.modules if name == package_name or name.startswith(f"{package_name}.")]:
            del sys.modules[module_name]


WORKSPACE_ROOT = REPO_ROOT.parent
prefer_local_repo("quant_orchestrator", WORKSPACE_ROOT / "quant-orchestrator")
prefer_local_repo("quant_warehouse", WORKSPACE_ROOT / "quant-warehouse")
prefer_local_repo("fmpsdk", WORKSPACE_ROOT / "fmpsdk")

from app.quant_warehouse_storage import ensure_quant_warehouse_storage
from app.trading_app_v2_runtime import (
    DEFAULT_STRATEGY_SOURCES,
    alpaca_client_from_env,
    build_alpaca_equity_orders,
    build_alpaca_option_orders,
    build_latest_equity_leaderboard,
    build_option_ml_ranking_table,
    backfill_thetadata_eod_for_score_date,
    backfill_thetadata_for_oracle_trade_windows,
    build_score_date_option_ml_ranking_table,
    build_symbol_score_table,
    build_robinhood_option_orders,
    build_llm_review_orders,
    build_llm_ranked_option_orders,
    build_ranked_alpaca_option_orders,
    default_paths,
    load_equity_artifacts,
    save_live_artifacts,
    select_optionable_leaderboard,
    write_streamlit_leaderboard_app,
)

paths = default_paths(REPO_ROOT)
paths.artifact_root.mkdir(parents=True, exist_ok=True)
paths.live_artifact_dir.mkdir(parents=True, exist_ok=True)
paths


## Environment

This notebook assumes `quant-warehouse`, `quant-orchestrator`, and `tradingagents` are already installed in the `optimal_trader` environment. Dependency installation belongs in environment setup, not in the live trading notebook.


In [ ]:
storage = ensure_quant_warehouse_storage()
display(storage.as_dict())


def optional_date(value):
    if value is None:
        return None
    text = str(value).strip()
    if not text or text.lower() in {"none", "null", "nat"}:
        return None
    return text


## Run Configuration

Shared knobs for universe screening, FMP refresh, feature engineering, target engineering, and downstream training. Feature-family and oracle-label plans are shown after warehouse data is refreshed.


In [ ]:
RUN_EQUITY_MOE_TRAINING = os.getenv("TRADING_APP_V2_RUN_EQUITY_TRAINING", "1") == "1"
RUN_OPTION_RANKER_TRAINING = os.getenv("TRADING_APP_V2_RUN_OPTION_TRAINING", "1") == "1"
DATA_START = "1900-01-01"
DATA_END = ""  # Empty means latest available downstream data.
MIN_MARKET_CAP = int(os.getenv("TRADING_APP_V2_MIN_MARKET_CAP", "10000000000"))
TOP_K = 20  # Use 10 or 20 for live candidate breadth.
MIN_LONG_SCORE = 0.50
REQUIRE_ALL_REQUESTED_FEATURE_FAMILIES = True
REFRESH_MISSING_FMP_DATA = os.getenv("TRADING_APP_V2_REFRESH_FMP", "1") == "1"
REFRESH_MISSING_FMP_INCLUDE_MACRO = True
REFRESH_MISSING_FMP_INCLUDE_PRICES = True
REFRESH_MISSING_FMP_STALENESS_DAYS = 60
REFRESH_MISSING_FMP_SKIP_RECENT_HOURS = 24.0
REFRESH_MISSING_FMP_MAX_WORKERS = 8
REFRESH_THETADATA_SCORE_DATE_EOD = os.getenv("TRADING_APP_V2_REFRESH_OPTIONS", "1") == "1"
THETADATA_OPTION_SCORE_DATE = ""  # Empty means latest equity score_date.
THETADATA_REFRESH_MAX_WORKERS = int(os.getenv("TRADING_APP_V2_OPTION_REFRESH_WORKERS", "1"))
THETADATA_REFRESH_OVERWRITE = False
REFRESH_THETADATA_ORACLE_TRADE_WINDOWS = os.getenv("TRADING_APP_V2_REFRESH_OPTION_TRAINING", "1") == "1"
THETADATA_ORACLE_BACKFILL_MAX_TRADES = int(os.getenv("TRADING_APP_V2_OPTION_BACKFILL_MAX_TRADES", "25"))
THETADATA_ORACLE_BACKFILL_WINDOW_DAYS = 7
THETADATA_ORACLE_FALLBACK_WINDOW_DAYS = 1
THETADATA_ORACLE_BACKFILL_REQUEST_SLEEP = 0.0
THETADATA_ORACLE_BACKFILL_OVERWRITE = False
RUN_BACKTESTS_IN_ORCHESTRATOR = False

ALPACA_EQUITY_ACCOUNT_PREFIX = "EQUITY"
ALPACA_OPTION_ACCOUNT_PREFIX = "OPTION"
ALPACA_LLM_ACCOUNT_PREFIX = "LLM"

OPTION_STRATEGY_ALLOCATION = 100_000.0
OPTION_BUCKET = "otm_option"
OPTION_TENOR_DAYS = 90

# Real Robinhood follows the option strategy with deeply discounted GTC limits.
ROBINHOOD_OPTION_GATE_DISCOUNT_PCT = 90.0

# Equity and LLM real-account gates remain blocked until paper PnL justifies changing them.
ROBINHOOD_EQUITY_GATE_DISCOUNT_PCT = 100.0
ROBINHOOD_LLM_GATE_DISCOUNT_PCT = 100.0

OPTION_PANEL_FOR_RANKER = paths.artifact_root / "option_candidate_panel.parquet"

display(
    {
        "artifact_root": str(paths.artifact_root),
        "equity_artifact_dir": str(paths.equity_artifact_dir),
        "option_artifact_dir": str(paths.option_artifact_dir),
        "live_artifact_dir": str(paths.live_artifact_dir),
        "data_start": DATA_START,
        "min_market_cap": MIN_MARKET_CAP,
        "refresh_missing_fmp_data": REFRESH_MISSING_FMP_DATA,
    }
)


## FMP Historical Refresh

Refresh raw FMP warehouse data before feature engineering and target engineering. This cell screens the universe first, displays the exact symbols that will be used downstream, then refreshes only those symbols in the shared Quant Warehouse store. The training cell below disables internal refresh so model fitting starts only after this visible refresh step completes.


In [ ]:
from quant_warehouse.migrate.backfill_missing_fmp import backfill_missing_fmp_historical
from quant_warehouse.research_tools.feature_family_eval import FamilyEvaluationConfig, screen_fmp_equity_universe
from quant_warehouse.warehouse.api import Warehouse

refresh_warehouse = Warehouse()
refresh_feature_config = FamilyEvaluationConfig(
    provider="fmp",
    market_cap_min=MIN_MARKET_CAP,
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
)

screened_equity_symbols, raw_fmp_universe, universe_eligibility, universe_source = screen_fmp_equity_universe(
    refresh_feature_config,
    warehouse=refresh_warehouse,
)
screened_symbols_df = pd.DataFrame({"symbol": list(screened_equity_symbols)})
print(
    f"FMP universe source={universe_source}; "
    f"eligible_symbols={len(screened_equity_symbols):,}; "
    f"min_market_cap={MIN_MARKET_CAP:,}"
)
display(screened_symbols_df)
print("Universe eligibility sample")
display(universe_eligibility.head(100))


def notebook_refresh_log(message: str) -> None:
    print(f"[fmp-refresh] {message}", flush=True)


fmp_refresh_summary = {"status": "skipped_by_config", "equity_symbols": list(screened_equity_symbols), "etf_symbols": []}
if REFRESH_MISSING_FMP_DATA:
    fmp_refresh_summary = backfill_missing_fmp_historical(
        warehouse=refresh_warehouse,
        equity_provider="fmp",
        etf_provider="fmp",
        equity_symbols=screened_equity_symbols,
        etf_symbols=(),
        include_macro=REFRESH_MISSING_FMP_INCLUDE_MACRO,
        include_prices=REFRESH_MISSING_FMP_INCLUDE_PRICES,
        staleness_days=REFRESH_MISSING_FMP_STALENESS_DAYS,
        skip_recent_hours=REFRESH_MISSING_FMP_SKIP_RECENT_HOURS,
        max_workers=REFRESH_MISSING_FMP_MAX_WORKERS,
        progress_logger=notebook_refresh_log,
    )
else:
    print("FMP refresh skipped because REFRESH_MISSING_FMP_DATA=False")

def macro_refresh_status(macro_payload) -> str | None:
    if isinstance(macro_payload, dict):
        return macro_payload.get("status")
    if isinstance(macro_payload, list):
        statuses = [
            str(item.get("status"))
            for item in macro_payload
            if isinstance(item, dict) and item.get("status") is not None
        ]
        if not statuses:
            return None
        counts = pd.Series(statuses).value_counts()
        return ", ".join(f"{name}={count}" for name, count in counts.items())
    return None


fmp_refresh_summary_path = paths.equity_artifact_dir / "fmp_refresh_summary.json"
fmp_refresh_summary_path.parent.mkdir(parents=True, exist_ok=True)
fmp_refresh_summary_path.write_text(json.dumps(fmp_refresh_summary, indent=2, default=str), encoding="utf-8")
print(f"Wrote FMP refresh summary to {fmp_refresh_summary_path}")
display(pd.DataFrame([{
    "equity_symbols": len(fmp_refresh_summary.get("equity_symbols", [])),
    "etf_symbols": len(fmp_refresh_summary.get("etf_symbols", [])),
    "equity_price_total": fmp_refresh_summary.get("equity_prices", {}).get("total"),
    "equity_price_updated": fmp_refresh_summary.get("equity_prices", {}).get("updated"),
    "equity_fundamental_total": fmp_refresh_summary.get("equity", {}).get("total"),
    "equity_fundamental_updated": fmp_refresh_summary.get("equity", {}).get("updated"),
    "macro_status": macro_refresh_status(fmp_refresh_summary.get("macro")),
}]))


## Target Engineering

Build collapsed oracle trade labels from refreshed warehouse prices. This notebook uses `oracle_only` labels: side-specific `YE` oracle entries for `k=1..12`, collapsed into `oracle_long` and `oracle_short` classes.


In [ ]:
from quant_warehouse.research_tools import BinaryTargetConfig
from quant_warehouse.platforms.data_providers.fmp.target_engineering import LabelBuildSpec, build_trade_results
from quant_warehouse.warehouse.api import Warehouse
from quant_orchestrator.research_tools.ml_trading_experiment import _build_oracle_trade_label_rows_sparse

if "screened_equity_symbols" not in globals():
    raise RuntimeError("Run the FMP Historical Refresh cell before target engineering.")

target_warehouse = Warehouse()

target_config = BinaryTargetConfig(
    provider="fmp",
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
    oracle_trade_k_by_frequency={"YE": tuple(range(1, 4))},
)

oracle_label_rows, oracle_label_diagnostics, oracle_seconds, oracle_trade_windows_unique = _build_oracle_trade_label_rows_sparse(
    screened_equity_symbols,
    target_config,
    warehouse=target_warehouse,
)

oracle_price_frames = {}
for raw_symbol in screened_equity_symbols:
    symbol = str(raw_symbol).strip().upper()
    if not symbol:
        continue
    prices = target_warehouse.read_prices(
        symbol,
        provider=target_config.provider,
        start=target_config.start_date,
        end=target_config.end_date,
    )
    if prices is not None and not prices.empty:
        oracle_price_frames[symbol] = prices

oracle_trade_spec = LabelBuildSpec(
    k_params={frequency: list(ks) for frequency, ks in target_config.oracle_trade_k_by_frequency.items()},
    min_profit_pct=target_config.oracle_trade_min_profit_pct,
    buy_execution=target_config.oracle_trade_long_entry_price_col,
    sell_execution=target_config.oracle_trade_long_exit_price_col,
    short_execution=target_config.oracle_trade_short_entry_price_col,
    cover_execution=target_config.oracle_trade_short_exit_price_col,
    start_date=target_config.start_date,
    end_date=target_config.end_date,
)
oracle_trade_result = build_trade_results(
    screened_equity_symbols,
    spec=oracle_trade_spec,
    price_frames=oracle_price_frames,
)
oracle_trade_windows = pd.DataFrame(oracle_trade_result.completed_trades)
if not oracle_trade_windows.empty:
    oracle_trade_windows["symbol"] = oracle_trade_windows["symbol"].astype(str).str.upper()
    oracle_trade_windows["entry_date"] = pd.to_datetime(oracle_trade_windows["entry_date"], errors="coerce").dt.normalize()
    oracle_trade_windows["exit_date"] = pd.to_datetime(oracle_trade_windows["exit_date"], errors="coerce").dt.normalize()
    oracle_trade_windows = oracle_trade_windows.dropna(subset=["symbol", "entry_date", "exit_date"])
    if "trade_id" not in oracle_trade_windows.columns:
        oracle_trade_windows["trade_id"] = (
            oracle_trade_windows["symbol"].astype(str)
            + "|"
            + oracle_trade_windows["side"].astype(str)
            + "|"
            + oracle_trade_windows["entry_date"].dt.strftime("%Y%m%d")
            + "|"
            + oracle_trade_windows["exit_date"].dt.strftime("%Y%m%d")
            + "|k"
            + oracle_trade_windows["k"].astype(str)
        )
    oracle_trade_windows = oracle_trade_windows.sort_values(
        ["entry_date", "symbol", "trade_id"],
        ascending=[False, True, True],
        kind="stable",
    ).reset_index(drop=True)

print(
    {
        "oracle_label_rows": len(oracle_label_rows),
        "oracle_classes": sorted(oracle_label_rows["collapsed_label"].dropna().unique())
        if not oracle_label_rows.empty
        else [],
        "oracle_seconds": round(oracle_seconds, 3),
        "oracle_trade_windows": len(oracle_trade_windows),
    }
)
display(oracle_label_diagnostics)
display(oracle_label_rows.head(20))

## ThetaData Oracle Entry/Exit Backfill

Backfill ThetaData EOD option chains only for oracle trade entry and exit dates from the current in-memory target-engineering table, newest entries first. Each endpoint date is checked against the Arctic cache before any ThetaData request is made.


In [ ]:
from quant_warehouse.migrate.backfill_thetadata_options import normalize_oracle_trade_windows

if "oracle_trade_windows" not in globals() or oracle_trade_windows.empty:
    raise RuntimeError("Run the Target Engineering cell before ThetaData oracle entry/exit backfill.")

required_oracle_trade_cols = {"symbol", "entry_date", "exit_date"}
missing_oracle_trade_cols = sorted(required_oracle_trade_cols.difference(oracle_trade_windows.columns))
if missing_oracle_trade_cols:
    raise RuntimeError(
        "The in-memory oracle_trade_windows table does not contain the columns needed for ThetaData entry/exit refresh: "
        f"{missing_oracle_trade_cols}"
    )

oracle_backfill_symbols = tuple(screened_equity_symbols) if "screened_equity_symbols" in globals() else ()
theta_oracle_trade_windows_raw = oracle_trade_windows.copy()
theta_oracle_trade_windows = normalize_oracle_trade_windows(
    theta_oracle_trade_windows_raw,
    symbols=oracle_backfill_symbols,
)
if theta_oracle_trade_windows.empty:
    raise RuntimeError("No normalized oracle trade windows are available for ThetaData entry/exit refresh.")

endpoint_pairs = pd.concat(
    [
        theta_oracle_trade_windows[["symbol", "entry_date"]].rename(columns={"entry_date": "snapshot_date"}),
        theta_oracle_trade_windows[["symbol", "exit_date"]].rename(columns={"exit_date": "snapshot_date"}),
    ],
    ignore_index=True,
)
endpoint_pairs["symbol"] = endpoint_pairs["symbol"].astype(str).str.upper()
endpoint_pairs["snapshot_date"] = pd.to_datetime(endpoint_pairs["snapshot_date"], errors="coerce").dt.normalize()
endpoint_pairs = endpoint_pairs.dropna(subset=["symbol", "snapshot_date"]).drop_duplicates()

theta_backfill_shape = {
    "raw_oracle_trade_rows": len(theta_oracle_trade_windows_raw),
    "deduped_symbol_entry_exit_windows": len(theta_oracle_trade_windows),
    "unique_symbol_endpoint_dates": len(endpoint_pairs),
    "symbols": int(theta_oracle_trade_windows["symbol"].nunique()),
}
print("ThetaData oracle entry/exit source: in-memory oracle_trade_windows")
display(pd.DataFrame([theta_backfill_shape]))
print(
    "ThetaData oracle entry/exit mode checks the Arctic cache first; "
    "ThetaData is called only for missing unique symbol/date entry-exit endpoints."
)

print("Historical ThetaData backfill is deferred until after live order artifacts are written.")


## Feature Engineering

Build the point-in-time feature-family panel from the refreshed Quant Warehouse store. This uses the screened FMP universe and keeps only the strategy sources configured for the equity MoE.


In [ ]:
from quant_warehouse.research_tools import (
    FamilyEvaluationConfig,
    build_fundamental_feature_panel,
    build_technical_feature_panel,
    cap_features_by_quality,
)
from quant_warehouse.warehouse.api import Warehouse

if "screened_equity_symbols" not in globals():
    raise RuntimeError("Run the FMP Historical Refresh cell before feature engineering.")

engineering_warehouse = Warehouse()
feature_config = FamilyEvaluationConfig(
    provider="fmp",
    market_cap_min=MIN_MARKET_CAP,
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
)

TECHNICAL_STRATEGY_SOURCES = (
    "fmp.technical_candles",
    "fmp.technical_cycles",
    "fmp.technical_math",
    "fmp.technical_momentum",
    "fmp.technical_overlap",
    "fmp.technical_performance",
)
training_observation_dates = oracle_label_rows[["symbol", "date"]].copy()
latest_observation_rows = []
for symbol, prices in oracle_price_frames.items():
    if prices is not None and not prices.empty:
        latest_observation_rows.append({"symbol": symbol, "date": pd.to_datetime(prices.index, errors="coerce").max()})
training_observation_dates = pd.concat(
    [training_observation_dates, pd.DataFrame(latest_observation_rows)],
    ignore_index=True,
).dropna().drop_duplicates()
raw_feature_panel, raw_feature_metadata, feature_diagnostics, feature_timings = build_fundamental_feature_panel(
    screened_equity_symbols,
    feature_config,
    warehouse=engineering_warehouse,
    strategy_sources=DEFAULT_STRATEGY_SOURCES,
    observation_dates=training_observation_dates,
)
technical_panel, technical_metadata, technical_diagnostics, technical_timings = build_technical_feature_panel(
    screened_equity_symbols,
    feature_config,
    strategy_sources=TECHNICAL_STRATEGY_SOURCES,
    observation_dates=training_observation_dates,
    warehouse=engineering_warehouse,
    max_workers=8,
    ta_mode="curated",
)
raw_feature_panel = raw_feature_panel.merge(technical_panel, on=["symbol", "date"], how="outer", validate="one_to_one")
raw_feature_metadata = pd.concat([raw_feature_metadata, technical_metadata], ignore_index=True).drop_duplicates()
feature_diagnostics = pd.concat([feature_diagnostics, technical_diagnostics], ignore_index=True, sort=False)
feature_timings = {**feature_timings, **technical_timings}
selected_features, selected_feature_metadata, feature_quality = cap_features_by_quality(
    raw_feature_panel,
    raw_feature_metadata,
    max_features=None,
)
wanted_sources = {str(source).strip() for source in (*DEFAULT_STRATEGY_SOURCES, *TECHNICAL_STRATEGY_SOURCES)}
raw_metadata = raw_feature_metadata.copy()
raw_metadata["strategy_source"] = raw_metadata["source"].astype(str) + "." + raw_metadata["family"].astype(str)
raw_available_sources = set(raw_metadata["strategy_source"].astype(str))
missing_raw_sources = sorted(wanted_sources.difference(raw_available_sources))
metadata = selected_feature_metadata.copy()
metadata["strategy_source"] = metadata["source"].astype(str) + "." + metadata["family"].astype(str)
selected_feature_metadata = (
    metadata.loc[metadata["strategy_source"].isin(wanted_sources)]
    .drop(columns=["strategy_source"])
    .reset_index(drop=True)
)
selected_available_sources = set((selected_feature_metadata["source"].astype(str) + "." + selected_feature_metadata["family"].astype(str)))
missing_selected_sources = sorted(wanted_sources.difference(selected_available_sources))
if REQUIRE_ALL_REQUESTED_FEATURE_FAMILIES and (missing_raw_sources or missing_selected_sources):
    raise RuntimeError(
        "Requested feature families are missing and equity ML training is configured to require all of them. "
        f"missing_raw={missing_raw_sources}; missing_selected={missing_selected_sources}"
    )
selected_features = [
    feature
    for feature in selected_features
    if feature in set(selected_feature_metadata["feature"].astype(str))
]
feature_panel = raw_feature_panel[["symbol", "date", *selected_features]].copy()
feature_panel["symbol"] = feature_panel["symbol"].astype(str).str.upper()
feature_panel["date"] = pd.to_datetime(feature_panel["date"], errors="coerce").dt.normalize()

print(
    {
        "symbols": len(screened_equity_symbols),
        "raw_feature_panel_rows": len(raw_feature_panel),
        "raw_feature_columns": len(raw_feature_panel.columns) if not raw_feature_panel.empty else 0,
        "selected_feature_columns": len(selected_features),
        "selected_feature_families": selected_feature_metadata["family"].nunique(),
        "requested_feature_families": len(wanted_sources),
        "missing_requested_raw_families": missing_raw_sources,
        "missing_requested_selected_families": missing_selected_sources,
        **feature_timings,
    }
)
display(feature_diagnostics.head(20))
display(
    selected_feature_metadata.groupby(["source", "family"], as_index=False)
    .agg(feature_count=("feature", "nunique"))
    .sort_values(["source", "family"])
)


## Equity MoE Training

Train RAPIDS/cuML random-forest classifiers on the refreshed warehouse data and engineered feature/target panels. This uses the screened `MIN_MARKET_CAP = 1_000_000_000_000` universe, fits on all available labeled rows from `1900-01-01` through the latest stored data, and writes the standard live-ranking artifacts. Internal orchestrator refresh, model probability diagnostics, and trading-validation diagnostics are disabled for this live-training path.


In [ ]:
from time import perf_counter
from types import SimpleNamespace

from quant_warehouse.lineage import build_dataset_lineage_manifest, write_dataset_lineage_manifest
from quant_orchestrator.research_tools import (
    FamilyClassifierConfig,
    FamilyScoreStore,
    ScoreMaterializationConfig,
    EquityMetaStackConfig,
    iter_feature_family_batches,
    train_and_materialize_family_scores,
    train_equity_meta_stack,
    write_ml_trading_artifact_files,
)

required_training_inputs = ["screened_equity_symbols", "feature_panel", "selected_feature_metadata", "oracle_label_rows"]
missing_training_inputs = [name for name in required_training_inputs if name not in globals()]
if missing_training_inputs:
    raise RuntimeError(
        "Run the FMP Historical Refresh, Feature Engineering, and Target Engineering cells before training. "
        f"Missing: {missing_training_inputs}"
    )

latest_feature_score_start = DATA_START
train_end_value = optional_date(DATA_END) or str(pd.Timestamp.today().date())
classifier_config = FamilyClassifierConfig(
    train_end=train_end_value,
    score_start=latest_feature_score_start,
    min_feature_coverage=0.50,
    min_train_rows=250,
    min_classes=2,
    random_seed=20260702,
    run_diagnostics=False,
)
family_score_output_dir = paths.equity_artifact_dir / "family_score_run"
lineage_dir = paths.equity_artifact_dir / "lineage"
lineage_cutoff = pd.Timestamp(train_end_value)
feature_lineage = build_dataset_lineage_manifest(
    feature_panel,
    dataset_id="trading_app_v2_feature_panel",
    dataset_kind="feature_panel",
    provider="fmp",
    available_at_cutoff=lineage_cutoff,
    recipe_id="quant_warehouse_feature_family_panel_v1",
    recipe={
        "strategy_sources": list((*DEFAULT_STRATEGY_SOURCES, *TECHNICAL_STRATEGY_SOURCES)),
        "min_market_cap": int(MIN_MARKET_CAP),
        "start_date": DATA_START,
        "end_date": optional_date(DATA_END),
    },
    source_references={"warehouse": str(storage.as_dict())},
)
label_lineage = build_dataset_lineage_manifest(
    oracle_label_rows,
    dataset_id="trading_app_v2_oracle_labels",
    dataset_kind="target_panel",
    provider="fmp",
    available_at_cutoff=lineage_cutoff,
    recipe_id="oracle_side_ye_k1_3_v1",
    recipe={"frequency": "YE", "k_min": 1, "k_max": 3},
    source_references={"warehouse": str(storage.as_dict())},
)
feature_lineage_path = write_dataset_lineage_manifest(
    feature_lineage, lineage_dir / f"features-{feature_lineage.lineage_fingerprint.removeprefix('sha256:')}.json"
)
label_lineage_path = write_dataset_lineage_manifest(
    label_lineage, lineage_dir / f"labels-{label_lineage.lineage_fingerprint.removeprefix('sha256:')}.json"
)
score_config = ScoreMaterializationConfig(
    output_dir=family_score_output_dir,
    run_id="trading_app_v2_live",
    target_id="oracle_side_ye_k1_3",
    input_lineage_paths=(feature_lineage_path, label_lineage_path),
    rows_per_chunk=250_000,
    persist_models=True,
    overwrite=True,
)

actual_training_families = (
    selected_feature_metadata[["source", "family"]]
    .drop_duplicates()
    .assign(model_id=lambda frame: frame["source"].astype(str) + "." + frame["family"].astype(str))
    .sort_values(["source", "family"])
    .reset_index(drop=True)
)
planned_models = actual_training_families["model_id"].tolist()
requested_sources = set(str(source).strip() for source in (*DEFAULT_STRATEGY_SOURCES, *TECHNICAL_STRATEGY_SOURCES))
missing_requested_sources = sorted(requested_sources.difference(planned_models))
unexpected_training_sources = sorted(set(planned_models).difference(requested_sources))
if REQUIRE_ALL_REQUESTED_FEATURE_FAMILIES:
    assert not missing_requested_sources, missing_requested_sources
    assert not unexpected_training_sources, unexpected_training_sources

training_plan = actual_training_families.assign(
    job=range(1, len(actual_training_families) + 1),
    estimator="RapidsRandomForestClassifier",
    representation="raw",
    lifecycle="train_score_persist_release",
)
print("Classifier-only family stream")
display(training_plan)
print("Training config")
display(asdict(classifier_config))
print("Score materialization config")
display({**asdict(score_config), "output_dir": str(score_config.output_dir)})
if missing_requested_sources:
    print("Requested model sources without selected feature columns")
    display(pd.DataFrame({"model_id": missing_requested_sources}))

if RUN_EQUITY_MOE_TRAINING:
    training_started = perf_counter()
    family_score_result = train_and_materialize_family_scores(
        iter_feature_family_batches(feature_panel, selected_feature_metadata),
        oracle_label_rows,
        classifier=classifier_config,
        materialization=score_config,
        progress_logger=lambda message: print(message, flush=True),
    )
    score_store = FamilyScoreStore(family_score_output_dir / "scores")
    family_scores = score_store.read_scores()
    equity_meta_result = train_equity_meta_stack(
        family_scores,
        oracle_label_rows,
        EquityMetaStackConfig(
            output_dir=paths.equity_artifact_dir / "equity_meta_stack",
            min_train_rows=classifier_config.min_train_rows,
            random_seed=classifier_config.random_seed,
        ),
    )
    strategy_scores = equity_meta_result.scores.copy()
    model_results = family_score_result.model_results.copy()
    elapsed_seconds = perf_counter() - training_started
    phase_timings = pd.DataFrame(
        [{"phase": "stream_family_train_score", "seconds": elapsed_seconds, "elapsed_seconds": elapsed_seconds}]
    )
    empty_diagnostics = pd.DataFrame()
    equity_metrics = {
        "symbols": int(strategy_scores["symbol"].nunique()) if not strategy_scores.empty else 0,
        "trained_models": int(model_results["status"].eq("ok").sum()) if "status" in model_results else 0,
        "strategy_sources": int(strategy_scores["strategy_source"].nunique()) if not strategy_scores.empty else 0,
        "score_rows": int(len(strategy_scores)),
        "elapsed_seconds": float(elapsed_seconds),
    }
    equity_result = SimpleNamespace(
        model_results=model_results,
        strategy_scores=strategy_scores,
        backtest_summary=empty_diagnostics,
        trade_log=empty_diagnostics,
        model_vs_trading=empty_diagnostics,
        metric_correlations=empty_diagnostics,
        yearly_backtest_summary=empty_diagnostics,
        symbol_strategy_summary=empty_diagnostics,
        symbol_robustness_summary=empty_diagnostics,
        backtesting_py_symbol_validation=empty_diagnostics,
        phase_timings=phase_timings,
        analysis_markdown="Classifier-only streaming training; each family was persisted and released before the next.",
        metrics=equity_metrics,
    )
    write_ml_trading_artifact_files(
        model_results=equity_result.model_results,
        strategy_scores=equity_result.strategy_scores,
        backtest_summary=equity_result.backtest_summary,
        trade_log=equity_result.trade_log,
        model_vs_trading=equity_result.model_vs_trading,
        metric_correlations=equity_result.metric_correlations,
        yearly_backtest_summary=equity_result.yearly_backtest_summary,
        symbol_strategy_summary=equity_result.symbol_strategy_summary,
        symbol_robustness_summary=equity_result.symbol_robustness_summary,
        backtesting_py_symbol_validation=equity_result.backtesting_py_symbol_validation,
        phase_timings=equity_result.phase_timings,
        analysis_markdown=equity_result.analysis_markdown,
        directory=paths.equity_artifact_dir,
    )
    live_config = {
        "classifier": asdict(classifier_config),
        "score_materialization": {**asdict(score_config), "output_dir": str(score_config.output_dir)},
        "quant_warehouse_storage": storage.as_dict(),
        "fmp_refresh_summary_path": str(fmp_refresh_summary_path),
    }
    (paths.equity_artifact_dir / "live_config.json").write_text(
        json.dumps(live_config, indent=2, default=str), encoding="utf-8"
    )
    print(equity_result.metrics)

if "equity_result" not in globals():
    cached_equity = load_equity_artifacts(paths.equity_artifact_dir)
    equity_result = SimpleNamespace(**cached_equity)
    family_score_result = SimpleNamespace(manifest_path=family_score_output_dir / "family_score_manifest.json")

strategy_scores = equity_result.strategy_scores.copy()
family_scores = FamilyScoreStore(family_score_output_dir / "scores").read_scores()
model_results = equity_result.model_results.copy()
backtest_summary = equity_result.backtest_summary.copy()

print("Actual classifier model rows")
display(model_results.head(50))
print("Latest reusable strategy scores")
display(strategy_scores.head())
print("Family score manifest")
print(family_score_result.manifest_path)


## Strategy Configuration

Review the 10B universe training configuration after model fitting: requested feature families, oracle `YE` labels for `k=1..12`, classifier jobs, and GPU preflight metadata.


In [ ]:
feature_family_plan = pd.DataFrame(
    [
        {
            "strategy_source": name,
            "provider": name.split(".", 1)[0],
            "feature_family": name.split(".", 1)[1],
        }
        for name in (*DEFAULT_STRATEGY_SOURCES, *TECHNICAL_STRATEGY_SOURCES)
    ]
)

oracle_label_plan = pd.DataFrame(
    [
        {
            "oracle_frequency": "YE",
            "k": k,
            "source_target_column": f"target_oracle_trade_entry__YE_k{k}_{side}",
            "classifier_label": "oracle_long" if side == "long" else "oracle_short" if side == "short" else "not a classifier class",
            "used_by_classifier": side in {"long", "short"},
        }
        for k in range(1, 4)
        for side in ("long", "short", "any")
    ]
)

classifier_plan = feature_family_plan.assign(
    classifier="RapidsRandomForestClassifier",
    backend="rapids_cuml_gpu",
    target_column="collapsed_label",
    classifier_labels="oracle_long, oracle_short",
    feature_representation="raw",
)

from quant_orchestrator.platforms.ml_frameworks.rapids.random_forest import rapids_gpu_info

gpu_preflight = rapids_gpu_info()

print("Feature families requested")
display(feature_family_plan)
print("Oracle label columns requested from YE k=1..3")
display(oracle_label_plan)
print("Classifier jobs planned")
display(classifier_plan)
print("RAPIDS GPU preflight")
display(gpu_preflight)


## Live Leaderboard

The leaderboard converts the current in-memory `strategy_scores` table into a live ranking. Latest prices are read from `quant-warehouse`, not from local `optimal_trader` data builders.

In [ ]:
leaderboard = build_latest_equity_leaderboard(
    strategy_scores,
    top_k=TOP_K,
    min_long_score=MIN_LONG_SCORE,
    price_provider="fmp",
)
display(leaderboard.head(TOP_K + 5))

## Option Ranker Training

This cell shows the direct `quant-orchestrator` option-family ranker call. It only reads the existing ThetaData-derived option candidate panel; it does not refresh or rebuild ThetaData option chains. The ranker joins FMP/FinanceToolkit feature-family panels from the shared `quant-warehouse` store and writes selector/family ranking artifacts for the option strategy.


In [ ]:
from quant_orchestrator.research_tools import (
    OptionMetaRankerConfig,
    train_option_meta_ranker,
)

option_ranker_config = OptionMetaRankerConfig(
    option_panel=OPTION_PANEL_FOR_RANKER.expanduser().resolve(),
    equity_score_store=family_score_output_dir / "scores",
    output_dir=paths.option_artifact_dir / "option_meta_stack",
    symbols=tuple(screened_equity_symbols),
    target_col="rank_y",
    n_estimators=300,
    max_trades=0,
    random_seed=20260704,
    max_candidates_per_trade=64,
)

display(asdict(option_ranker_config))

if RUN_OPTION_RANKER_TRAINING:
    if not OPTION_PANEL_FOR_RANKER.exists():
        raise FileNotFoundError(
            f"Missing option panel {OPTION_PANEL_FOR_RANKER}. ThetaData refresh/build is intentionally disabled here; point OPTION_PANEL_FOR_RANKER at an existing candidate panel."
        )
    option_ranker_result = train_option_meta_ranker(option_ranker_config)
    print(option_ranker_result.output_dir)
    display({"training_rows": option_ranker_result.training_rows, "training_trades": option_ranker_result.training_trades, "features": len(option_ranker_result.features)})
else:
    print("Option ranker training skipped; not reading old option ranker summary CSV files.")


## ThetaData Score-Date EOD Refresh

Refresh the ThetaData EOD option chain only for the current selected equity symbols and the equity score date. This keeps live option ML ranking scoped to contracts that were tradable on the score date instead of historical expired option panels.


In [ ]:
if "leaderboard" not in globals() or leaderboard.empty:
    raise RuntimeError("Run the leaderboard cell before ThetaData score-date refresh.")

theta_score_date = optional_date(THETADATA_OPTION_SCORE_DATE)
if theta_score_date is None:
    theta_score_date = pd.to_datetime(leaderboard["score_date"], errors="coerce").max().strftime("%Y-%m-%d")

option_leaderboard = select_optionable_leaderboard(leaderboard, score_date=theta_score_date, top_k=TOP_K)
theta_selected_symbols = option_leaderboard["symbol"].astype(str).str.upper().tolist()
print(f"ThetaData score-date EOD date: {theta_score_date}")
print(f"ThetaData selected symbols: {theta_selected_symbols}")

thetadata_score_date_refresh_summary = None
if REFRESH_THETADATA_SCORE_DATE_EOD:
    thetadata_score_date_refresh_summary = backfill_thetadata_eod_for_score_date(
        symbols=theta_selected_symbols,
        score_date=theta_score_date,
        max_workers=THETADATA_REFRESH_MAX_WORKERS,
        overwrite=THETADATA_REFRESH_OVERWRITE,
    )
    display(thetadata_score_date_refresh_summary)
else:
    print("ThetaData score-date refresh skipped because REFRESH_THETADATA_SCORE_DATE_EOD=False")

score_date_option_ml_rankings = build_score_date_option_ml_ranking_table(
    paths.option_artifact_dir,
    leaderboard=option_leaderboard,
    score_date=theta_score_date,
    symbols=theta_selected_symbols,
    target_dte=OPTION_TENOR_DAYS,
    min_market_cap=MIN_MARKET_CAP,
    start_date=DATA_START,
    equity_family_scores=family_scores,
)
print(f"Score-date option ML ranking rows: {len(score_date_option_ml_rankings)}")
display(score_date_option_ml_rankings)


## Build Broker Order Plans

Three Alpaca paper accounts are managed separately:

- equity paper: equity variant of the feature-family MoE strategy
- option paper: option variant of the same equity signals
- LLM paper: top trades sent to `trading_agents` for review before paper execution

Robinhood real orders are based on the option strategy and use GTC limit orders. Option buys are priced at the bid and option sells are priced at the ask; the Robinhood gate controls whether new orders are allowed.

In [ ]:
paper_order_plans: dict[str, pd.DataFrame] = {}
paper_order_clients: dict[str, object] = {}
option_plan: dict[str, pd.DataFrame] = {}

try:
    equity_orders = build_alpaca_equity_orders(
        leaderboard=leaderboard,
        account_prefix=ALPACA_EQUITY_ACCOUNT_PREFIX,
        gross_exposure=0.95,
    )
    paper_order_plans["alpaca_equity_paper"] = equity_orders
    paper_order_clients["alpaca_equity_paper"] = alpaca_client_from_env(ALPACA_EQUITY_ACCOUNT_PREFIX)
except Exception as exc:
    print(f"Alpaca equity paper plan skipped: {type(exc).__name__}: {exc}")

try:
    option_orders = build_ranked_alpaca_option_orders(
        option_rankings=score_date_option_ml_rankings,
        decisions=option_leaderboard[["symbol", "direction"]],
        account_prefix=ALPACA_OPTION_ACCOUNT_PREFIX,
    )
    paper_order_plans["alpaca_option_paper"] = option_orders
    robinhood_targets = score_date_option_ml_rankings.loc[
        score_date_option_ml_rankings["selected_by_option_ensemble"].astype(bool)
    ].merge(option_leaderboard[["symbol", "direction"]], on="symbol", how="inner")
    robinhood_targets = robinhood_targets.loc[
        ((robinhood_targets["direction"] == "long") & (robinhood_targets["option_type"] == "call"))
        | ((robinhood_targets["direction"] == "short") & (robinhood_targets["option_type"] == "put"))
    ].copy()
    robinhood_targets["quantity"] = 1
    robinhood_targets["expiry_date"] = robinhood_targets["expiration"]
    robinhood_targets["strike_price"] = robinhood_targets["strike"]
    robinhood_targets["combined_score"] = robinhood_targets["pred_meta_stack_rank"]
    robinhood_targets["bid_price"] = pd.to_numeric(robinhood_targets.get("bid"), errors="coerce")
    robinhood_targets["ask_price"] = pd.to_numeric(robinhood_targets.get("ask"), errors="coerce")
    option_plan = {"target_contracts": robinhood_targets}
    paper_order_clients["alpaca_option_paper"] = alpaca_client_from_env(ALPACA_OPTION_ACCOUNT_PREFIX)
except Exception as exc:
    print(f"Alpaca option paper plan skipped: {type(exc).__name__}: {exc}")

try:
    llm_orders, llm_reviews = build_llm_ranked_option_orders(
        leaderboard=option_leaderboard,
        option_rankings=score_date_option_ml_rankings,
        top_k=TOP_K,
        account_prefix=ALPACA_LLM_ACCOUNT_PREFIX,
    )
    paper_order_plans["alpaca_llm_paper"] = llm_orders
    paper_order_clients["alpaca_llm_paper"] = alpaca_client_from_env(ALPACA_LLM_ACCOUNT_PREFIX)
except Exception as exc:
    print(f"Alpaca LLM paper plan skipped: {type(exc).__name__}: {exc}")

for name, frame in paper_order_plans.items():
    print(name, len(frame))
    display(frame.head(20))

In [ ]:
robinhood_option_plan: dict[str, pd.DataFrame] = {}
robinhood_option_orders = pd.DataFrame()
if option_plan and not option_plan.get("target_contracts", pd.DataFrame()).empty:
    robinhood_option_plan = build_robinhood_option_orders(
        target_contracts=option_plan.get("target_contracts", pd.DataFrame()),
        gate_discount_pct=ROBINHOOD_OPTION_GATE_DISCOUNT_PCT,
        current_option_positions=pd.DataFrame(),
        pending_option_orders=pd.DataFrame(),
    )
    robinhood_option_orders = robinhood_option_plan.get("actionable_orders", pd.DataFrame())

if robinhood_option_plan:
    print("Robinhood option reconciliation summary")
    display(robinhood_option_plan.get("summary", pd.DataFrame()))
    print("Robinhood current option positions")
    display(robinhood_option_plan.get("current_option_positions", pd.DataFrame()).head(20))
    print("Robinhood pending option orders")
    display(robinhood_option_plan.get("pending_option_orders", pd.DataFrame()).head(20))

print("Robinhood actionable option orders")
display(robinhood_option_orders.head(20))
print({
    "robinhood_option_gate_discount_pct": ROBINHOOD_OPTION_GATE_DISCOUNT_PCT,
    "robinhood_equity_gate_discount_pct": ROBINHOOD_EQUITY_GATE_DISCOUNT_PCT,
    "robinhood_llm_gate_discount_pct": ROBINHOOD_LLM_GATE_DISCOUNT_PCT,
    "option_pricing_policy": "buy at bid, sell at ask",
})


## Save Leaderboard And Streamlit App

In [ ]:
option_ml_rankings_for_live = (
    score_date_option_ml_rankings
    if "score_date_option_ml_rankings" in globals()
    else build_option_ml_ranking_table(
        paths.option_artifact_dir,
        symbols=leaderboard.loc[leaderboard["selected"].astype(bool), "symbol"].tolist() if "selected" in leaderboard.columns else leaderboard["symbol"].tolist(),
    )
)
symbol_scores_for_live = build_symbol_score_table(strategy_scores, leaderboard)
orders_for_live = {**paper_order_plans, "robinhood_option_real": robinhood_option_orders}

import socket


def port_is_open(port: int, host: str = "127.0.0.1") -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.2)
        return sock.connect_ex((host, int(port))) == 0


def first_free_port(start: int = 8501, stop: int = 8599) -> int:
    for port in range(int(start), int(stop) + 1):
        if not port_is_open(port):
            return port
    raise RuntimeError(f"No free Streamlit port found between {start} and {stop}")


saved_live_paths = save_live_artifacts(
    live_dir=paths.live_artifact_dir,
    leaderboard=leaderboard,
    symbol_scores=symbol_scores_for_live,
    option_ml_rankings=option_ml_rankings_for_live,
    orders=orders_for_live,
)
if "llm_reviews" in globals():
    llm_reviews.to_csv(paths.live_artifact_dir / "trading_agents_reviews.csv", index=False)
streamlit_app = write_streamlit_leaderboard_app(
    live_dir=paths.live_artifact_dir,
    leaderboard=leaderboard,
    symbol_scores=symbol_scores_for_live,
    option_ml_rankings=option_ml_rankings_for_live,
    orders=orders_for_live,
)
streamlit_port = first_free_port()
print({"streamlit_app": str(streamlit_app), "data_source": "in_memory_embedded"})
if streamlit_port != 8501:
    print(f"Port 8501 is already in use; using free port {streamlit_port} for this app.")
print(f"Run: streamlit run {streamlit_app} --server.address 127.0.0.1 --server.port {streamlit_port}")
print(f"Streamlit URL: http://localhost:{streamlit_port}")
print(f"Streamlit URL: http://127.0.0.1:{streamlit_port}")

# Non-critical-path historical progress: orders and frontend already exist above.
thetadata_oracle_trade_window_summary = {"status": "skipped_by_config"}
if REFRESH_THETADATA_ORACLE_TRADE_WINDOWS:
    try:
        thetadata_oracle_trade_window_summary = backfill_thetadata_for_oracle_trade_windows(
            theta_oracle_trade_windows,
            symbols=oracle_backfill_symbols,
            max_trades=THETADATA_ORACLE_BACKFILL_MAX_TRADES,
            backfill_window_days=THETADATA_ORACLE_BACKFILL_WINDOW_DAYS,
            fallback_window_days=THETADATA_ORACLE_FALLBACK_WINDOW_DAYS,
            overwrite=THETADATA_ORACLE_BACKFILL_OVERWRITE,
            request_sleep=THETADATA_ORACLE_BACKFILL_REQUEST_SLEEP,
        )
    except Exception as exc:
        thetadata_oracle_trade_window_summary = {
            "status": "failed_after_live_artifacts",
            "error": f"{type(exc).__name__}: {exc}",
        }
else:
    print("Historical ThetaData backfill skipped by configuration.")
historical_backfill_summary_path = paths.live_artifact_dir / "thetadata_historical_backfill_summary.json"
historical_backfill_summary_path.write_text(
    json.dumps(thetadata_oracle_trade_window_summary, indent=2, default=str), encoding="utf-8"
)
display(thetadata_oracle_trade_window_summary)

## Submit Orders

The notebook never submits orders. Review the saved order plans in the generated Streamlit app, then use its Submit Orders button when you intentionally want to send broker orders.
